# Lifecycle model server

Serve one paper model on a Colab GPU and expose an OpenAI-compatible endpoint that the
Lifecycle direct adapter can reach from a laptop.

The engine stays on the laptop. Only model inference runs here, so the transaction
journal, evidence, and decisions never leave the local machine.

Before running: Runtime, Change runtime type, and select a GPU.

## 1. Confirm the GPU

Stop here if this reports no GPU. A CPU Colab runtime is slower than the laptop.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Install Ollama and start it

Ollama already speaks the OpenAI chat completions API at `/v1`, which is exactly what
the direct adapter expects, so no translation layer is needed.

In [ ]:
import subprocess, time, urllib.request

# The Ollama archive is zstd-compressed and Colab does not ship zstd.
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# Ollama defaults to a 4096-token context. The Wiki Maintainer receives up to
# 8 traces of 15,000 characters each, so raise it before the server starts.
import os
server_env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="32768")
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=server_env,
)

for attempt in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
        print("ollama is serving")
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("ollama did not start")

## 3. Pull the model

Paper models, in order of appetite. A 16 GB GPU holds the 9B comfortably at 4-bit.

| model | approximate size | note |
|---|---|---|
| `qwen3.5:4b` | 3.4 GB | smallest paper model, fastest iteration |
| `qwen3.5:9b` | about 6 GB | the recommended first real run |
| `qwen3.5:27b` | about 17 GB | needs an A100 or L4 |

In [ ]:
MODEL = "qwen3.5:9b"

subprocess.run(["ollama", "pull", MODEL], check=True)
subprocess.run(["ollama", "list"], check=True)

## 4. Measure throughput

Record the real number before planning iterations. The first request includes model load
and GPU warm-up, so its prompt rate is not representative; run it twice and use the second.

Measured 2026-09-23 on a T4 with `qwen3.5:9b`, warm: 24.8 generated and 202.3 prompt tokens
per second. The laptop baseline was 1.5 generated tokens per second on `qwen3.5:4b`.

In [ ]:
import json

payload = json.dumps({
    "model": MODEL,
    "prompt": "Answer with only the letter. Which is prime? A. 4 B. 6 C. 7 D. 8 E. 9",
    "stream": False,
    "options": {"num_predict": 128},
}).encode()

request = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=payload,
    headers={"Content-Type": "application/json"},
)
result = json.loads(urllib.request.urlopen(request, timeout=600).read())

generated = result.get("eval_count", 0) / (result.get("eval_duration", 1) / 1e9)
prompt_rate = result.get("prompt_eval_count", 0) / (result.get("prompt_eval_duration", 1) / 1e9)
print(f"generated: {generated:.1f} tok/s")
print(f"prompt:    {prompt_rate:.1f} tok/s")
print(f"total:     {result.get('total_duration', 0) / 1e9:.1f} s")

## 5. Expose the endpoint

The laptop needs to reach this runtime. Paste a Cloudflare tunnel token or use the
quick tunnel below, then give the printed URL plus `/v1` to the adapter.

Treat the printed URL as a secret while the run lasts: anyone holding it can send
requests to this runtime. Stop the tunnel when the run ends.

In [ ]:
import re

subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True,
    check=True,
)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        public_url = found.group(0)
        break

print("endpoint for the adapter:")
print(f"  {public_url}/v1")

## 6. Run the loop from the laptop

On the laptop, point the driver at the printed endpoint:

```
python drive_baseline.py \
  --domain livemath --domain-root <path> --phase baseline \
  --model qwen3.5:9b --provider ollama \
  --base-url https://<printed>.trycloudflare.com/v1
```

Keep this tab open. A Colab runtime stops when the browser disconnects for long enough,
and the tunnel dies with it. The engine's transaction journal survives an interruption,
so a dropped run resumes rather than corrupting state, but every completed task must be
recorded before the runtime ends.